# 03 — World-line preview (illustrative only)

**Status: `ILLUSTRATIVE PREVIEW - NOT VALIDATED`.** This is a communication and debugging artifact, not a result. It is read-only and CPU only; no fitting.

- **Two steps only.** Step 1 is sampled from the frozen one-step model. Step 2 uses a **state-lag-only approximation**, not a generative feature model.
- **No multi-step accuracy claim.** No test labels are read and no accuracy is computed.
- **Actor states are projections.** Node cells count active sampled dyads, not full-network activity. They are not node-level predictions.
- **Deletable.** Removing this preview does not affect any main result, table, figure, or abstract sentence.


In [ ]:
from pathlib import Path
import json

import pandas as pd

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 200)

try:
    from IPython.display import display
except Exception:
    display = print


def find_repo_root() -> Path:
    here = Path.cwd().resolve()
    for cand in (here, *here.parents):
        if (cand / "results" / "framework_pilot").is_dir():
            return cand
    return here


ROOT = find_repo_root()
WORLDLINE = ROOT / "results" / "worldline_preview"


def rel(path: Path) -> str:
    try:
        return str(path.relative_to(ROOT))
    except ValueError:
        return str(path)


def load_json(path: Path) -> dict:
    if not path.is_file():
        print(f"[missing] {rel(path)}")
        return {}
    with path.open(encoding="utf-8") as fh:
        return json.load(fh)


def load_csv(path: Path) -> pd.DataFrame:
    if not path.is_file():
        print(f"[missing] {rel(path)}")
        return pd.DataFrame()
    return pd.read_csv(path)


print("repo root:", ROOT)
print("worldline artifacts:", "present" if WORLDLINE.is_dir() else "MISSING")


## Anchor summary

Four anchors from the inner-validation period (2016-01 to 2017-11). The sample is selected for visibility, not random: five source actors at evenly spaced activity ranks, with a small set of active and inactive dyads per source.


In [ ]:
meta = load_json(WORLDLINE / "worldline_trace.json")

if meta:
    print("status:", meta.get("status"))
    print("disclaimer:", meta.get("disclaimer"))
    print("period:", meta.get("period", {}).get("name"))
    print("model:", meta.get("model", {}).get("name"))
    print("sample rule:", meta.get("sample_selection", {}).get("rule"))

    anchors = pd.DataFrame(meta.get("anchors", []))
    if not anchors.empty:
        anchors["p1_mean"] = anchors["p1_summary"].apply(
            lambda d: d.get("mean")
        )
        anchors["p2_mean"] = anchors["p2_summary"].apply(
            lambda d: d.get("mean")
        )
        atab = anchors[
            [
                "anchor", "dates", "n_dyads", "observed_active",
                "sampled_active_step1", "sampled_active_step2",
                "p1_mean", "p2_mean",
            ]
        ].rename(
            columns={
                "anchor": "Anchor",
                "dates": "Dates",
                "n_dyads": "Sampled dyads",
                "observed_active": "Observed active",
                "sampled_active_step1": "Active at step 1",
                "sampled_active_step2": "Active at step 2",
                "p1_mean": "Mean p(t+1)",
                "p2_mean": "Mean p(t+2)",
            }
        )
        for col in ["Mean p(t+1)", "Mean p(t+2)"]:
            atab[col] = atab[col].astype(float).round(4)
        display(atab)
    else:
        print("Anchor summaries not available.")
else:
    print("World-line metadata not available.")


## Preview figure

The PNG is stored under `results/worldline_preview/`, so it is part of the public artifact release. It shows two representative anchors; the trace covers all four.


In [ ]:
figure = WORLDLINE / "worldline_preview.png"

if not figure.is_file():
    print(f"[missing] {rel(figure)}")
else:
    try:
        from IPython.display import Image, display as show_image
    except Exception as exc:
        Image = None
        show_image = print
        print(f"[note] IPython display unavailable: {exc}")

    if Image is not None:
        show_image(Image(filename=str(figure)))
    else:
        print(f"Figure available at {rel(figure)}")


## Trace excerpt

One row per anchor × sampled dyad × step. Step 0 is observed; step 1 uses the frozen one-step model; step 2 uses the state-lag-only approximation. `p_model` is blank at step 0 by construction.


In [ ]:
trace = load_csv(WORLDLINE / "worldline_trace.csv")

if not trace.empty:
    excerpt = trace[trace["anchor"] == "2016-06"][
        [
            "anchor", "step", "date", "source_actor", "target_actor",
            "sample_role", "state", "state_origin", "p_model",
            "sim_n_events",
        ]
    ].head(12)
    display(excerpt)
else:
    print("World-line trace CSV not available.")


## Aggregate transitions

Counts of dyad and induced-actor state transitions (00, 01, 10, 11) over the sampled subset. These are descriptive counts for the preview, not estimated transition rates.


In [ ]:
if meta:
    records = []
    for unit, steps in meta.get("aggregate_transitions", {}).items():
        for step, counts in steps.items():
            for transition, n in counts.items():
                records.append(
                    {
                        "unit": unit,
                        "step": step,
                        "transition": transition,
                        "n": n,
                    }
                )
    if records:
        ttab = (
            pd.DataFrame(records)
            .pivot(index=["unit", "step"], columns="transition", values="n")
            .reset_index()
        )
        display(ttab)
    else:
        print("Aggregate transitions not available.")
else:
    print("World-line metadata not available.")


## Induced actor states

`active` means the actor has at least one active sampled dyad. `n_active_sampled_dyads` is a count within the 20-dyad sample, not full-network activity. These rows are explanation-side projections, not node-level forecasts.


In [ ]:
nodes = load_csv(WORLDLINE / "node_trace.csv")

if not nodes.empty:
    ntab = nodes[nodes["anchor"] == "2016-06"][
        [
            "anchor", "step", "role", "actor", "active",
            "n_active_sampled_dyads", "transition",
        ]
    ].head(12)
    display(ntab)
else:
    print("Node trace CSV not available.")


## What this does not show

- No accuracy, calibration, or multi-step validation.
- No random sample and no prevalence inference.
- No uncertainty over sampled paths; the preview is one path per anchor.
- No generative update of graph, reverse-dyad, actor-ego, or train-only features at step 2; those are frozen at the anchor.
- No node-level forecast claim. Actor states are induced by the sampled dyad subset only.

A validated rolling claim would need an intermediate-month feature-generating model, out-of-sample scoring at h = 1, 2, …, paired cluster-bootstrap intervals, and a pre-registered error-accumulation stopping rule. None of that is present here.
